In [1]:
import pandas as pd
import numpy as np
import ta
from datetime import datetime

from lib.config import PAIRES, TIMEFRAMES, FULL_NAMES
from lib.data import get_cached_data
from lib.analysis import (
    detect_candlestick_patterns,
    find_support_resistance,
    analyze_market_structure,
    build_price_action_signal,
)
from lib.ai import analyze_with_ai

In [2]:
# Pretelchargement de toutes les paires / timeframes
print("=" * 60)
print("TELECHARGEMENT DES DONNEES")
print("=" * 60)

data = {}
for pair in PAIRES:
    print("\n" + pair)
    print("-" * 40)
    pair_data = {}
    for tf_name, tf in TIMEFRAMES.items():
        symbol = pair + tf["suffix"]
        print(f"  {tf_name:6s} ({tf['interval']:4s} / {tf['period']:4s})")
        pair_data[tf_name] = get_cached_data(symbol, tf['interval'], tf['period'])
    data[pair] = pair_data

TELECHARGEMENT DES DONNEES

EURUSD
----------------------------------------
  daily  (1d   / 1mo )
✗ Cache expired (age: 1222m)
⬇ Downloading fresh data for EURUSD=X...
  h1     (1h   / 1mo )
✗ Cache expired (age: 1048m)
⬇ Downloading fresh data for EURUSD=X...
  intra  (5m   / 1d  )
✗ Cache expired (age: 1222m)
⬇ Downloading fresh data for EURUSD=X...

USDJPY
----------------------------------------
  daily  (1d   / 1mo )
✗ Cache expired (age: 1222m)
⬇ Downloading fresh data for USDJPY=X...
  h1     (1h   / 1mo )
✗ Cache expired (age: 1046m)
⬇ Downloading fresh data for USDJPY=X...
  intra  (5m   / 1d  )
✗ Cache expired (age: 1222m)
⬇ Downloading fresh data for USDJPY=X...

GBPUSD
----------------------------------------
  daily  (1d   / 1mo )
✗ Cache expired (age: 1222m)
⬇ Downloading fresh data for GBPUSD=X...
  h1     (1h   / 1mo )
✗ Cache expired (age: 1053m)
⬇ Downloading fresh data for GBPUSD=X...
  intra  (5m   / 1d  )
✗ Cache expired (age: 1222m)
⬇ Downloading fresh data for G

In [3]:
# Analyse technique de toutes les paires
results = []

for pair in PAIRES:
    df_daily = data[pair]["daily"]
    df_h1    = data[pair]["h1"]
    df_intra = data[pair]["intra"]

    # --- Indicateurs daily ---
    df_daily["rsi"] = ta.momentum.RSIIndicator(close=df_daily["Close"], window=14).rsi()
    macd = ta.trend.MACD(close=df_daily["Close"])
    df_daily["macd"] = macd.macd()
    df_daily["macd_signal"] = macd.macd_signal()
    df_daily["ma20"] = df_daily["Close"].rolling(20).mean()

    last_d = df_daily.iloc[-1]
    daily_rsi = round(float(last_d["rsi"]), 1)
    daily_macd = "bullish" if last_d["macd"] > last_d["macd_signal"] else "bearish"
    daily_trend = "uptrend" if float(last_d["Close"]) > float(last_d["ma20"]) else "downtrend"

    # --- Structure et niveaux ---
    daily_structure = analyze_market_structure(df_daily)
    daily_levels = find_support_resistance(df_daily)
    struct_1h = analyze_market_structure(df_h1)
    levels_1h = find_support_resistance(df_h1)

    # --- Intraday ---
    patterns = detect_candlestick_patterns(df_intra)
    levels_5m = find_support_resistance(df_intra)
    struct_5m = analyze_market_structure(df_intra)
    signal = build_price_action_signal(df_intra, patterns, levels_5m, struct_5m)

    # Enrichir
    signal["pair"] = pair + " (" + FULL_NAMES[pair] + ")"
    signal["price"] = round(float(df_intra["Close"].iloc[-1]), 5)
    signal['daily_rsi'] = daily_rsi
    signal['daily_macd'] = daily_macd
    signal['daily_trend'] = daily_trend
    signal['daily_bias'] = daily_structure['bias']
    signal['h1_bias'] = struct_1h['bias']
    signal['daily_support'] = [l['level'] for l in daily_levels if l['type'] == 'Support'][:2]
    signal['daily_resistance'] = [l['level'] for l in daily_levels if l['type'] == 'Resistance'][:2]
    signal['h1_support'] = [l['level'] for l in levels_1h if l['type'] == 'Support'][:2]
    signal['h1_resistance'] = [l['level'] for l in levels_1h if l['type'] == 'Resistance'][:2]

    results.append(signal)

In [4]:
# Tableau recapitulatif
print("=" * 80)
print("                SCAN MULTI-PAIRES")
print("=" * 80)

header = f'{"Paire":<20} {"Prix":>10} {"RSI":>6} {"MACD":>8} {"Trend D":>10} {"Bias D":>10} {"Bias 1H":>10}'
print(header)
print("-" * 80)

for s in results:
    r = s['daily_rsi']
    m = s['daily_macd'].upper()
    t = s['daily_trend'].upper()
    db = s['daily_bias']
    hb = s['h1_bias']
    line = f'{s["pair"]:<20} {s["price"]:>10.5f} {r:>6} {m:>8} {t:>10} {db:>10} {hb:>10}'
    print(line)

                SCAN MULTI-PAIRES
Paire                      Prix    RSI     MACD    Trend D     Bias D    Bias 1H
--------------------------------------------------------------------------------
EURUSD (EUR/USD)        1.15128   63.9  BEARISH    UPTREND    Neutral    Neutral
USDJPY (USD/JPY)      156.96700   30.4  BEARISH  DOWNTREND    Bullish    Neutral
GBPUSD (GBP/USD)        1.34347   55.6  BEARISH    UPTREND    Neutral    Neutral


In [5]:
# Analyse IA pour chaque paire
print("=" * 80)
print("                ANALYSE IA")
print("=" * 80)

for i, s in enumerate(results):
    print('\n' + s['pair'])
    print("-" * 60)

    ai_result = analyze_with_ai(s)

    if ai_result:
        emojis = {'BUY': '[BUY]', 'SELL': '[SELL]', 'HOLD': '[HOLD]'}
        ai_signal = ai_result.get('signal', 'UNKNOWN').upper()
        conf = ai_result.get('confidence', 0)

        print(f"  Signal:       {ai_signal} {emojis.get(ai_signal, '')}")
        print(f"  Confiance:    {conf}%")
        print(f"  Raison:       {ai_result.get('reason', 'N/A')}")

        entry = ai_result.get('entry')
        sl = ai_result.get('stop_loss')
        tp = ai_result.get('take_profit')
        if entry is not None and sl is not None and tp is not None:
            try:
                entry, sl, tp = float(entry), float(sl), float(tp)
                print(f"  Entree:       {entry:.5f}")
                print(f"  Stop Loss:    {sl:.5f}")
                print(f"  Take Profit:  {tp:.5f}")
            except (ValueError, TypeError):
                print(f"  Entree:       {entry}")
                print(f"  Stop Loss:    {sl}")
                print(f"  Take Profit:  {tp}")
    else:
        print("  Analyse IA indisponible")

                ANALYSE IA

EURUSD (EUR/USD)
------------------------------------------------------------
  Signal:       HOLD [HOLD]
  Confiance:    65%
  Raison:       The market is in consolidation (ranging) with a Doji pattern, indicating indecision. Although the daily trend is an uptrend, the current short-term bias and price action suggest waiting for a breakout or clear directional move.

USDJPY (USD/JPY)
------------------------------------------------------------
  Signal:       HOLD [HOLD]
  Confiance:    45%
  Raison:       Conflicting signals: Short-term (London session) is bullish with strong support nearby, but daily indicators (RSI/MACD/Trend) are bearish/downtrend. Wait for confirmation or a clear breakout.

GBPUSD (GBP/USD)
------------------------------------------------------------
  Signal:       HOLD [HOLD]
  Confiance:    65%
  Raison:       The market is in consolidation (ranging) during the London session, indicated by the Doji pattern and neutral bias across mu